# Spotify Music Clustering Analysis
Bu notebook'ta Spotify şarkı özelliklerini (danceability, energy, vb.) inceleyip **K-Means Clustering** kullanarak şarkıları türlerine (veya ses profillerine) göre otomatik olarak kümeleyeceğiz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Grafik ayarları
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#121212", "figure.facecolor": "#121212", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})

## 1. Veri Yükleme ve Ön İnceleme

In [ ]:
df = pd.read_csv('spotify_data.csv')
display(df.head())
display(df.info())

## 2. Keşifsel Veri Analizi (EDA)
Özelliklerin (features) birbirleriyle olan ilişkilerine bakalım.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='energy', y='loudness', hue='genre_label', palette='viridis', alpha=0.7)
plt.title('Enerji vs Ses Yüksekliği (Gerçek Türlere Göre)')
plt.show()

## 3. Veri Ön İşleme (Preprocessing)
Makine öğrenmesi modelinin düzgün çalışabilmesi için sayısal verileri standartlaştırıyoruz (Scaling).

In [ ]:
features = ['danceability', 'energy', 'acousticness', 'loudness', 'tempo', 'valence']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 4. K-Means Kümeleme (Clustering)
K-Means modelini kurup şarkıları özelliklerine göre 5 kümeye ayırıyoruz.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Hangi kümede kaç şarkı var?
display(df['cluster'].value_counts())

## 5. Sonuçların Görselleştirilmesi (PCA ile Boyut İndirgeme)
6 boyutlu veriyi 2 boyuta (PCA) indirgeyip kümeleri çizdirelim.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='cluster', palette='rainbow', s=70, alpha=0.8)
plt.title('Spotify Şarkıları Kümeleme Sonuçları (PCA 2D)')
plt.show()

## 6. Özet ve Sonuçlar
### Q&A
- **Soru:** Şarkıları etiketleri olmadan benzer özelliklerine göre gruplayabilir miyiz?
- **Cevap:** Evet, K-Means ile enerji, tempo ve akustiklik gibi özelliklere göre akustik şarkılar bir kümede, enerjik EDM'ler başka bir kümede toplandı.

### Data Analysis Key Findings
- Model, gerçek tür etiketlerini (genre_label) görmese bile aynı türe ait şarkıların ses profillerini tanıyarak kendi içinde tutarlı kümeler oluşturdu.
- Enerji ve ses yüksekliği (loudness) pozitif korelasyona sahip.

### Insights or Next Steps
- Kümelerin içindeki şarkı isimlerini inceleyerek bir **Şarkı Tavsiye Sistemi (Recommendation Engine)** kurulabilir.
- Farklı model algoritmaları (DBSCAN vs) denenerek veya özellik çıkarımı artırılarak başarı oranı artırılabilir.